# Docling as MCP tool with Llama Stack

### Overview

In this lab... we will:
- Get familiar with Docling MCP server 
- Chat with LM Studio
- RAG with LlamaStack
- Information extraction with LlamaStack + LM Studio + Pyantic AI

---


### Technologies We'll Use

Building on our previous labs, we will leverage:

1. **[Docling](https://docling-project.github.io/docling/):** An open-source toolkit used to parse and convert documents.
2. **[MCP](https://modelcontextprotocol.io)**: The model context protocol for creating a tool.
3. **[Llama Stack](https://llama-stack.readthedocs.io/)**: Framework for building generative AI applications.
4. **[Pydantic AI](https://ai.pydantic.dev/)**: Another framework for building generative AI applications.
5. **[LM Studio](https://lmstudio.ai/)**: An AI platform to run local LLMs on your computer.

---

## Prerequisites

Before we begin, ensure you have:
- Followed the instructions in [README.md](../README.md) to run this workshop notebooks. In particular, ensure to create a virtual environment, for instance, with [uv](https://docs.astral.sh/uv/), and install the `llama-stack` optional dependencies, as well as the `examples` group dependencies:
  ```bash
  uv venv
  source .venv/bin/activate
  uv sync --extra llama-stack --group examples
  ```
- Python >=3.10 installed
- [Podman](https://podman.io/docs/installation) installed (or Docker)
- [LM Studio](https://lmstudio.ai/) installed and running on port 1234

---

## Installation and Setup


#### Notebook dependencies

Create a virtual environment to run this notebook, for instance, with [uv](https://docs.astral.sh/uv/), and install the necessary dependencies:

In [1]:
! uv pip install llama-stack-client==0.2.18 pydantic pydantic_settings rich python-dotenv

Using Python 3.12.10 environment at: /Users/ceb/git/docling-workshops/workshops/2025_08_29/.venv
Resolved 37 packages in 292ms                                        
Installed 5 packages in 14ms.2.18                           
 + distro==1.9.0
 + fire==0.7.1
 + llama-stack-client==0.2.18
 + pyaml==25.7.0
 + termcolor==3.1.0


#### Docling MCP server

Run **Docling MCP** with the tools for _conversion_, _generation_, _llama-stack-rag_. In a new terminal run:

```shell
uvx --from docling-mcp --with llama-stack \
    docling-mcp-server \
    --transport streamable-http \
    --port 8000 \
    --host 0.0.0.0 \
    conversion generation llama-stack-rag
```

#### LM Studio

You can now install the **Docling MCP** server into **LM Studio**.

[LM Studio](https://lmstudio.ai/) supports MCP tools and allows to run your agentic workloads completely locally. The configuration is done by editing the `mcp.json` file (following the instructions [here](https://lmstudio.ai/docs/app/plugins/mcp)), or with the convenience button below:

[![Add MCP Server docling to LM Studio](https://files.lmstudio.ai/deeplink/mcp-install-light.svg)](https://lmstudio.ai/install-mcp?name=docling&config=eyJjb21tYW5kIjoidXZ4IiwiYXJncyI6WyItLWZyb209ZG9jbGluZy1tY3AiLCJkb2NsaW5nLW1jcC1zZXJ2ZXIiXX0%3D)

However, we will install the **local** MCP server that we ran in the previous step to leverage the Llama Stack additional capabilities.
Simply edit the `mcp.json` file with the following configuration:

```json
{
  "mcpServers": {
    "docling": {
      "url": "http://localhost:8000/mcp"
    }
  }
}
```

You can now explore the tools from Docling MCP throw the user interface:
- Check the list of docling tools on `Program > Integrations` and ensure they are enabled
- Download an language **Model**, for instance, `openai/gpt-oss-20b`
- Open a **Chat** and try different queries to levereage Docling capabilities:

  - 👤 Convert the document on https://arxiv.org/pdf/2408.09869 and give me a summary of the document
  - 👤 Generate a thumbnail from the page 1
  - 👤 Create a new Docling document with the title "Agentic AI", a paragraph, and a list of the main applications.

#### Llama Stack with LM Studio

Since [LM Studio](https://lmstudio.ai/) is exposing an openai-compatible api, we can use it as a local inference server.
Since Llama Stack does not have a native provider for it, we connect it as a vllm server.

1. Install and run [LM Studio](https://lmstudio.ai/). Ensure that the server is started on port 1234.

2. Run Llama Stack.

    ```shell
    export LLAMA_STACK_PORT=8321
 
    podman run \
        -it \
        --pull always \
        -p $LLAMA_STACK_PORT:$LLAMA_STACK_PORT \
        -v ~/.llama:/root/.llama \
        llamastack/distribution-starter:0.2.18 \
        --port $LLAMA_STACK_PORT \
        --env VLLM_URL=http://host.containers.internal:1234/v1 \
        --env MILVUS_URL=http://host.containers.internal:19530
    ```

3. Register models (should have been downloaded in LM Studio already).

    ```sh
    uvx --from llama-stack-client llama-stack-client models register lms/llama-3.2-3b-instruct --provider-id vllm --provider-model-id llama-3.2-3b-instruct

    uvx --from llama-stack-client llama-stack-client models register lms/openai/gpt-oss-20b --provider-id vllm --provider-model-id openai/gpt-oss-20b

    uvx --from llama-stack-client llama-stack-client models register lms/ibm/granite-3.2-8b --provider-id vllm --provider-model-id ibm/granite-3.2-8b
    ```

4. Test the models.

    ```sh
    uvx --from llama-stack-client llama-stack-client --endpoint http://localhost:8321 \
        inference chat-completion \
        --model-id lms/llama-3.2-3b-instruct \
        --message "Write a short story about a robot."
    ```

Now let's import the essential modules:

In [1]:
import logging
import uuid
import json
from json import JSONDecodeError

from llama_stack_client import Agent
from llama_stack_client import LlamaStackClient
from llama_stack_client.lib.agents.event_logger import EventLogger
from pydantic import NonNegativeFloat, AnyHttpUrl
from pydantic_settings import BaseSettings, SettingsConfigDict
from rich.pretty import pprint
from termcolor import cprint

# set the logger
logger = logging.getLogger(__name__)
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(message)s")
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


# access the environment variables
class Settings(BaseSettings):
    base_url: AnyHttpUrl = "http://localhost:8321"
    inference_model: str = "lms/llama-3.2-3b-instruct"
    max_tokens: int = 4096
    temperature: NonNegativeFloat = 0.0
    top_p: float = 0.95
    stream: bool = False
    use_prompt_chaining: bool = True

    docling_mcp_url: AnyHttpUrl = "http://host.containers.internal:8000/mcp"

    vdb_provider: str = "milvus"
    vdb_embedding: str = "all-MiniLM-L6-v2"
    vdb_embedding_dimension: int = 384
    # vdb_embedding_window: int = 256

    model_config = SettingsConfigDict(
        env_file=".env", env_file_encoding="utf-8", extra="ignore"
    )


settings = Settings()
pprint(settings)

def step_printer(steps):
    """
    Print the steps of an agent's response in a formatted way.
    Note: stream need to be set to False to use this function.
    Args:
    steps: List of steps from an agent's response.
    """
    for i, step in enumerate(steps):
        step_type = type(step).__name__
        print("\n" + "-" * 10, f"📍 Step {i + 1}: {step_type}", "-" * 10)
        if step_type == "ToolExecutionStep":
            print("🔧 Executing tool...")
            try:
                pprint(json.loads(step.tool_responses[0].content))
            except (TypeError, JSONDecodeError):
                # tool response is not a valid JSON object
                pprint(step.tool_responses[0].content)
        else:
            if step.api_model_response.content:
                print("🤖 Model Response:")
                cprint(f"{step.api_model_response.content}\n", "magenta")
            elif step.api_model_response.tool_calls:
                tool_call = step.api_model_response.tool_calls[0]
                print("🛠️ Tool call Generated:")
                cprint(
                    f"Tool call: {tool_call.tool_name}, "
                    f"Arguments: {json.loads(tool_call.arguments_json)}",
                    "magenta",
                )
    print("=" * 10, "Query processing completed", "=" * 10, "\n")


def user_printer(query: str) -> None:
    print("👤 User Query:")
    cprint(query, "cyan")

Settings(
│   base_url=AnyHttpUrl('http://localhost:8321/'),
│   inference_model='lms/llama-3.2-3b-instruct',
│   max_tokens=4096,
│   temperature=0.0,
│   top_p=0.95,
│   stream=False,
│   use_prompt_chaining=True,
│   docling_mcp_url=AnyHttpUrl('http://host.containers.internal:8000/mcp'),
│   vdb_provider='milvus',
│   vdb_embedding='all-MiniLM-L6-v2',
│   vdb_embedding_dimension=384
)

Establish the connection to your Llama Stack server.

In [2]:
client = LlamaStackClient(base_url=str(settings.base_url))
print(f"Connected to Llama Stack server @ {client.base_url}")

Connected to Llama Stack server @ http://localhost:8321/


Fetch the inference-related parameters from the corresponding environment variables and convert them to the format Llama Stack expects.

In [3]:
if settings.temperature > 0.0:
    strategy = {
        "type": "top_p",
        "temperature": settings.temperature,
        "top_p": settings.top_p,
    }
else:
    strategy = {"type": "greedy"}

# sampling_params will later be used to pass the parameters to Llama Stack Agents/Inference APIs
sampling_params = {
    "strategy": strategy,
    "max_tokens": settings.max_tokens,
}

print(
    f"Inference Parameters:\n\tSampling Parameters: {sampling_params}\n\tstream: {settings.stream}"
)

Inference Parameters:
	Sampling Parameters: {'strategy': {'type': 'greedy'}, 'max_tokens': 4096}
	stream: False


Validate that the Docling MCP tools are available in the Llama Stack instance.

When an instance of Llama Stack is redeployed, it may be the case that the tools will need to be re-registered. Also if a tool is already registered with a Llama Stack instance, trying to register another one with the same `toolgroup_id` will throw you an error.

For this reason, it is recommended to validate your tools and toolgroups. The following code will check that `mcp::docling` tools are correctly registered, and if not it will attempt to register them using their specific endpoints.

In [4]:
registered_tools = client.tools.list()
registered_toolgroups = [t.toolgroup_id for t in registered_tools]
if "mcp::docling" not in registered_toolgroups:
    client.toolgroups.register(
        toolgroup_id="mcp::docling",
        provider_id="model-context-protocol",
        mcp_endpoint={"uri": str(settings.docling_mcp_url)},
    )

registered_tools = client.tools.list()
registered_toolgroups = [t.toolgroup_id for t in registered_tools]
print(
    f"Your Llama Stack server is registered with the following tool groups @ {set(registered_toolgroups)} \n"
)

INFO:httpx:HTTP Request: GET http://localhost:8321/v1/tools "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/tools "HTTP/1.1 200 OK"


Your Llama Stack server is registered with the following tool groups @ {'mcp::docling', 'builtin::rag', 'builtin::websearch'} 



## Agentic RAG with Llama Stack

We will use tools internal to Llama Stack and from the Docling MCP server that allow executing tasks such as:
- [`mcp::docling`] converting a PDF file from a local or remote location into a unified document representation [DoclingDocument](https://docling-project.github.io/docling/concepts/docling_document/).
- [`mcp::docling`] chunk and ingest the document in the Llama Stack vectordb.
- [`builtin::rag/knowledge_search`] search in the document using agentic rag techniques.

We define an agent provided with the **Docling MCP** tools together with the built-in knowledge_search. The agent should be able to accomplish the following tasks in a multi-step, multi-tool approach:

1. Converting a PDF file into the `DoclingDocument` format.
2. Ingest the results in the vector db.
3. Search in the vector db using an agentic/multi-step approach.

In [5]:
model_prompt = """You are a helpful assistant. You have access to a number of tools.
Whenever a tool is called, be sure to return the Response in a friendly and helpful tone.
"""

In [6]:
# define the name of the vectordb collection to use
vector_db_id = f"test_vector_db_{uuid.uuid4()}"

# define and register the document collection to be used
client.vector_dbs.register(
    vector_db_id=vector_db_id,
    embedding_model=settings.vdb_embedding,
    embedding_dimension=settings.vdb_embedding_dimension,
    provider_id=settings.vdb_provider,
)


# Create simple agent with tools
agent = Agent(
    client=client,
    model=settings.inference_model,  # replace this with model_id to get the value of INFERENCE_MODEL_ID environment variable
    instructions=model_prompt,  # update system prompt based on the model you are using
    tools=[
        dict(
            name="mcp::docling/convert_document_into_docling_document",
            args={},
        ),
        dict(
            name="mcp::docling/insert_document_to_vectordb",
            args={
                "vector_db_id": vector_db_id,
            },
        ),
        dict(
            name="builtin::rag/knowledge_search",
            args={
                "vector_db_ids": [
                    vector_db_id
                ],  # list of IDs of document collections to consider during retrieval
            },
        ),
    ],
    tool_config={"tool_choice": "auto"},
    sampling_params=sampling_params,
)

INFO:httpx:HTTP Request: POST http://localhost:8321/v1/vector-dbs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/agents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/tools?toolgroup_id=mcp%3A%3Adocling%2Fconvert_document_into_docling_document "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/tools?toolgroup_id=mcp%3A%3Adocling%2Finsert_document_to_vectordb "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8321/v1/tools?toolgroup_id=builtin%3A%3Arag%2Fknowledge_search "HTTP/1.1 200 OK"


In [8]:
user_prompts = [
    "Convert the PDF document on https://arxiv.org/pdf/2206.01062 to DoclingDocument.",
    "Ingest the document.",
    "Answer with the document knowledge in the vectordb: How many pages were manually annotated in the dataset?",
]
session_id = agent.create_session(f"docling-session_{uuid.uuid4()}")

for i, prompt in enumerate(user_prompts):
    user_printer(prompt)
    response = agent.create_turn(
        messages=[{"role": "user", "content": prompt}],
        session_id=session_id,
        stream=settings.stream,
    )
    if settings.stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        step_printer(
            response.steps
        )  # print the steps of an agent's response in a formatted way.

INFO:llama_stack_client._base_client:Retrying request to /v1/agents/2fb10796-970b-490d-9059-fb1490a2a33d/session in 0.422785 seconds
INFO:llama_stack_client._base_client:Retrying request to /v1/agents/2fb10796-970b-490d-9059-fb1490a2a33d/session in 0.842748 seconds
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/agents/2fb10796-970b-490d-9059-fb1490a2a33d/session "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8321/v1/agents/2fb10796-970b-490d-9059-fb1490a2a33d/session/cf352b59-d407-4666-85a4-c77b963db139/turn "HTTP/1.1 200 OK"


👤 User Query:
Convert the PDF document on https://arxiv.org/pdf/2206.01062 to DoclingDocument.

---------- 📍 Step 1: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: convert_document_into_docling_document, Arguments: {'source': 'https://arxiv.org/pdf/2206.01062'}

---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


[
│   TextContentItem(
│   │   text='{\n  "from_cache": true,\n  "document_key": "868f49ae1f0e66e82238a8aea43fd30b"\n}',
│   │   type='text'
│   )
]


---------- 📍 Step 3: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: insert_document_to_vectordb, Arguments: {'document_key': '868f49ae1f0e66e82238a8aea43fd30b', 'vector_db_id': 'my_vector_db'}

---------- 📍 Step 4: ToolExecutionStep ----------
🔧 Executing tool...


[
│   TextContentItem(
│   │   text='{\n  "vector_db_id": "test_vector_db_2cd48449-9b52-493e-bef4-db25b611a1f7"\n}',
│   │   type='text'
│   )
]

...


---------- 📍 Step 5: InferenceStep ----------
🤖 Model Response:
The function calls have been made to convert the PDF document on https://arxiv.org/pdf/2206.01062 to DoclingDocument and then insert it into a vectordb. The output of the first function call is used as input for the second function call.

========== Query processing completed ========== 

👤 User Query:
Ingest the document.


INFO:httpx:HTTP Request: POST http://localhost:8321/v1/agents/2fb10796-970b-490d-9059-fb1490a2a33d/session/cf352b59-d407-4666-85a4-c77b963db139/turn "HTTP/1.1 200 OK"



---------- 📍 Step 1: InferenceStep ----------
🤖 Model Response:
This query does not require a function call. It can be answered directly. 

The document with the key "868f49ae1f0e66e82238a8aea43fd30b" has been ingested into the vectordb.

========== Query processing completed ========== 

👤 User Query:
Answer with the document knowledge in the vectordb: How many pages were manually annotated in the dataset?

---------- 📍 Step 1: InferenceStep ----------
🛠️ Tool call Generated:
Tool call: knowledge_search, Arguments: {'query': 'number of pages manually annotated in dataset'}

---------- 📍 Step 2: ToolExecutionStep ----------
🔧 Executing tool...


[
│   TextContentItem(
│   │   text='knowledge_search tool found 2 chunks:\nBEGIN of knowledge_search tool results.\n',
│   │   type='text'
│   ),
│   TextContentItem(
│   │   text="Result 1\nContent: 3 THE DOCLAYNET DATASET\nDocLayNet contains 80863 PDF pages. Among these, 7059 carry two instances of human annotations, and 1591 carry three. This amounts to 91104 total annotation instances. The annotations provide layout information in the shape of labeled, rectangular boundingboxes. We define 11 distinct labels for layout features, namely Caption , Footnote , Formula , List-item , Page-footer , Page-header , Picture , Section-header , Table , Text , and Title . Our reasoning for picking this particular label set is detailed in Section 4.\nIn addition to open intellectual property constraints for the source documents, we required that the documents in DocLayNet adhere to a few conditions. Firstly, we kept scanned documents\nFigure 2: Distribution of DocLayNet pages across document categories.\nMetadata: {'chunk_id': '7156212269791437020-00012', 'document_id': '7156212269791437020', 'source': None, 'doc_items': ['#/texts/339', '#/texts/340', '#/texts/343']}\n",
│   │   type='text'
│   ),
│   TextContentItem(
│   │   text="Result 2\nContent: 1 INTRODUCTION\n- (1) Human Annotation : In contrast to PubLayNet and DocBank, we relied on human annotation instead of automation approaches to generate the data set.\n- (2) Large Layout Variability : We include diverse and complex layouts from a large variety of public sources.\n- (3) Detailed Label Set : We define 11 class labels to distinguish layout features in high detail. PubLayNet provides 5 labels; DocBank provides 13, although not a superset of ours.\n- (4) Redundant Annotations : A fraction of the pages in the DocLayNet data set carry more than one human annotation.\n1 https://developer.ibm.com/exchanges/data/all/doclaynet\nThis enables experimentation with annotation uncertainty and quality control analysis.\n- (5) Pre-defined Train-, Test- & Validation-set : Like DocBank, we provide fixed train-, test- & validation-sets to ensure proportional representation of the class-labels. Further, we prevent leakage of unique layouts across sets, which has a large effect on model accuracy scores.\nMetadata: {'chunk_id': '7156212269791437020-00009', 'document_id': '7156212269791437020', 'source': None, 'doc_items': ['#/texts/326', '#/texts/327', '#/texts/328', '#/texts/329', '#/texts/330', '#/texts/331', '#/texts/332']}\n",
│   │   type='text'
│   ),
│   TextContentItem(text='END of knowledge_search tool results.\n', type='text'),
│   TextContentItem(
│   │   text='The above results were retrieved to help answer the user\'s query: "number of pages manually annotated in dataset". Use them as supporting information only in answering this query.\n',
│   │   type='text'
│   )
]


---------- 📍 Step 3: InferenceStep ----------
🤖 Model Response:
The document contains 91104 total annotation instances, which amounts to 7059 pages with two instances of human annotations and 1591 pages with three instances of human annotations.

========== Query processing completed ========== 



In [ ]:
import uuid
import logging

from llama_stack_client import LlamaStackClient
from pydantic import NonNegativeFloat
from pydantic_settings import BaseSettings, SettingsConfigDict

# pretty print of the results returned from the model/agent
from rich.console import Console
from rich.pretty import pprint

### Logging

To see detailed information about the document processing and chunking operations, we'll configure INFO log level.

NOTE: It is okay to skip running this cell if you prefer less verbose output.

In [ ]:
console = Console()

logger = logging.getLogger(__name__)
if not logger.hasHandlers():  
    logger.setLevel(logging.INFO)
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    formatter = logging.Formatter('%(message)s')
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

### Setting up

In the following blocks we setup the environment needed for connecting to llama stack and use it as agentic framework.

In [ ]:
class Settings(BaseSettings):
    base_url: str

    vdb_provider: str
    vdb_embedding: str
    vdb_embedding_dimension: int
    vdb_embedding_window: int

    inference_model_id: str
    max_tokens: int
    temperature: NonNegativeFloat
    top_p: float
    stream: bool

    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8")

In [ ]:
settings = Settings(
    base_url="http://localhost:8321",
    inference_model_id="meta-llama/Llama-3.2-3B-Instruct",
    max_tokens=4096,
    temperature=0.0,
    top_p=0.95,
    stream=True,
    vdb_provider="faiss",
    vdb_embedding="all-MiniLM-L6-v2",
    vdb_embedding_dimension=384,
    vdb_embedding_window=256,
)
print(settings)

In [ ]:
if settings.temperature > 0.0:
    strategy = {
        "type": "top_p",
        "temperature": settings.temperature,
        "top_p": settings.top_p,
    }
else:
    strategy = {"type": "greedy"}

# sampling_params will later be used to pass the parameters to Llama Stack Agents/Inference APIs
sampling_params = {
    "strategy": strategy,
    "max_tokens": settings.max_tokens,
}

print(sampling_params)

---

## Launch Llama Stack

Within this lab we will interact with a Llama Stack backend. We have chosen the Ollama distribution which allows to easily get started on a local environment.

### Fetch the models

In a terminal window use the following command for fetching the models required for running.

```bash
export INFERENCE_MODEL="meta-llama/Llama-3.2-3B-Instruct"

# ollama names this model differently, and we must use the ollama name when loading the model
export OLLAMA_INFERENCE_MODEL="llama3.2:3b-instruct-fp16"
ollama run $OLLAMA_INFERENCE_MODEL --keepalive 60m
  ```


### Start the Llama Stack container

In a new terminal window use the following command to run the Llama Stack server.

```bash
# make a working directory which will be used by the container
mkdir -p ~/.llama

# launch llama stack
export LLAMA_STACK_PORT=8321
podman run \
  -it \
  --pull always \
  -p $LLAMA_STACK_PORT:$LLAMA_STACK_PORT \
  -v ~/.llama:/root/.llama \
  llamastack/distribution-ollama \
  --port $LLAMA_STACK_PORT \
  --env INFERENCE_MODEL=$INFERENCE_MODEL \
  --env OLLAMA_URL=http://host.containers.internal:11434
```


Next we can use the `LlamaStackClient` within this notebook validate the connection.

In [ ]:
client = LlamaStackClient(base_url=settings.base_url)
print(f"Connected to Llama Stack server @ {client.base_url}")

---

## Launch the MCP tool

MCP allows to connect custom tools (like Docling) within an agentic framework. In this lab we will use an MCP tool which allows to
1. Convert documents using Docling
2. Ingest them into a Llama Stack vector DB instance.

_You can inspect how a tool is created by looking at the file [Docling_Lab4_tool.py](./Docling_Lab4_tool.py)_

We already packaged the tool into a working container image which is ready for you to try out.

**Launch the Docling Llama Stack MCP tool** by running the following command in a new terminal window.

```bash
podman run \
  -it \
  --pull always \
  -p 8000:8000 \
  quay.io/docling-project/lab-demo-docling-llamstack-mcp \
  --env DOCLING_MCP_LLAMA_STACK_URL=http://host.containers.internal:8321
```

---

### Validate tools available in our llama-stack instance

When an instance of llama-stack is redeployed your tools need to re-registered. Also if a tool is already registered with a llama-stack instance, if you try to register one with the same `toolgroup_id`, llama-stack will throw you an error.

For this reason it is recommended to include some code to validate your tools and toolgroups. This is where the `mcp_url` comes into play. The following code will check that the `mcp::docling-llamastack` tool is registered, or it will be registered directly from the mcp url.

If you are running the MCP server from source, the default value for this is: `http://localhost:8000/sse`.

If you are running the MCP server from a container, the default value for this is: `http://host.containers.internal:8000/sse`.

Make sure to pass the corresponding MCP URL for the server you are trying to register/validate tools for.

In [ ]:
docling_mcp_url = "http://host.containers.internal:8000/sse"

registered_tools = client.tools.list()
registered_toolgroups = [t.toolgroup_id for t in registered_tools]

if "mcp::docling-llamastack" not in registered_toolgroups:
    client.toolgroups.register(
        toolgroup_id="mcp::docling-llamastack",
        provider_id="model-context-protocol",
        mcp_endpoint={"uri":docling_mcp_url},
    )

registered_tools = client.tools.list()
registered_toolgroups = [t.toolgroup_id for t in registered_tools]
logger.info(f"Your Llama Stack server is already registered with the following tool groups @ {set(registered_toolgroups)} \n")

## Ingest + RAG-aware agent

- Initialize the collection in the vectordb
- Initialize the agent the required tools:
    - Docling Ingest will be responsible to take care of instructions like "Ingest the document https://arxiv.org/pdf/2503.11576".
    - RAG/Knowledge search will respond to user queries by running RAG on the documents ingested in the vectordb.

In [ ]:
from llama_stack_client import Agent, AgentEventLogger
from llama_stack_client.lib.agents.event_logger import EventLogger


In [ ]:
# define the name of the vectordb collection to use
vector_db_id = f"test_vector_db_{uuid.uuid4()}"

# define and register the document collection to be used
client.vector_dbs.register(
    vector_db_id=vector_db_id,
    embedding_model=settings.vdb_embedding,
    embedding_dimension=settings.vdb_embedding_dimension,
    provider_id=settings.vdb_provider,
)


agent = Agent(
    client,
    model=settings.inference_model_id,
    instructions="You are a helpful assistant.",
    sampling_params=sampling_params,
    tools=[
        dict(
            name="mcp::docling-llamastack",
            args={
                "vector_db_id": vector_db_id,
            },
        ),
        dict(
            name="builtin::rag/knowledge_search",
            args={
                "vector_db_ids": [vector_db_id],  # list of IDs of document collections to consider during retrieval
            },
        )
    ],

)

In [ ]:
v=client.vector_dbs.retrieve(vector_db_id)
client.vector_io.query(vector_db_id=vector_db_id, query="docling")

## Executing ingest and RAG queries

- For each prompt, initialize a new agent session, execute a turn during which a retrieval call may be requested, and output the reply received from the agent.

In [ ]:
queries = [
    "Ingest the document https://arxiv.org/pdf/2503.11576",
    "Lookup the documents to answer the question: How does the system compare to humans when analyzing the layout?",
]

for prompt in queries:
    console.print(f"\n[cyan]User> {prompt}[/cyan]")
    
    # create a new turn with a new session ID for each prompt
    response = agent.create_turn(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        session_id=agent.create_session(f"rag-session_{uuid.uuid4()}"),
        stream=settings.stream,
    )
    
    # print the response, including tool calls output
    if settings.stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        pprint(response.steps)

## What happened?

The code above executed a chat interaction with an agent.

With the first message, we instruct the agent to ingest the document. The model, performing its reasoning, plans decides to call the Docling tool for converting the document.

With the second message, we ask the agent to search the ingested content.
Note how the model decides on its own which one is a good query for search the relevant chunks in the vector database. Compared to the previous labs, the retrieval is not done with the exact use query. This is interpreted and tuned.

---

# ReAct Agent

In the following section we use the reasoning agent `ReActAgent`. In this scenario, the model orchestrator the tools execution is reasoning on the sequence of tools to be executed in order to perform the task.

This allows to have a single user query which triggers multiple independent steps, e.g.

1. Ingest the documents
2. Run a search query on the documents


In [ ]:
from llama_stack_client.lib.agents.react.agent import ReActAgent
from llama_stack_client.lib.agents.react.tool_parser import ReActOutput


In [ ]:
vector_db_id = f"test_vector_db_{uuid.uuid4()}"

# define and register the document collection to be used
client.vector_dbs.register(
    vector_db_id=vector_db_id,
    embedding_model=settings.vdb_embedding,
    embedding_dimension=settings.vdb_embedding_dimension,
    provider_id=settings.vdb_provider,
)



In [ ]:

agent = ReActAgent(
            client=client,
            model=settings.inference_model_id,
            tools=[
                dict(
                    name="mcp::docling-llamastack",
                    args={
                        "vector_db_id": vector_db_id,
                    },
                ),
                dict(
                    name="builtin::rag/knowledge_search",
                    args={
                        "vector_db_ids": [vector_db_id],  # list of IDs of document collections to consider during retrieval
                    },
                )
            ],
            response_format={
                "type": "json_schema",
                "json_schema": ReActOutput.model_json_schema(),
            },
            sampling_params=sampling_params,
        )
user_prompts = [
    "I would like to summarize the statements of the authors of https://arxiv.org/pdf/2503.11576 on how does SmolDocling compare to humans when analyzing the layout."
]

for prompt in user_prompts:
    print("\n"+"="*50)
    console.print(f"[cyan]Processing user query: {prompt}[/cyan]")
    print("="*50)
    response = agent.create_turn(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        session_id=agent.create_session(f"rag-session_{uuid.uuid4()}"),
        stream=settings.stream
    )
    if settings.stream:
        for log in EventLogger().log(response):
            log.print()
    else:
        pprint(response.steps) # print the steps of an agent's response in a formatted way. 

## What happened?

Compared to the previous agent, here the model used advanced reasining for creating a plan of actions needed to perform the operation.

---

# Summary and Next Steps

### What You've Accomplished

Congratulations! You've successfully used Docling in an agentic framework. Here's what you've learned:

- **Lab 1**: Document structure preservation enables everything else
- **Lab 2**: Intelligent chunking optimizes retrieval quality
- **Lab 3**: Visual grounding transforms RAG into transparent AI
- **Lab 4**: Run Docling as MCP tool with Llama Stack


## Next Steps: Where to Go from Here

### Immediate actions

1. **Experiment with your documents**
   - Try documents with complex layouts
   - Test with technical diagrams and charts
   - Process multi-page reports with mixed content

2. **Connect more agents**
   - Try connecting more tools
   - Search the documents to ingest via metadata
   - Search the web for relevant documents
   - Extract information from the documents

3. **More ways to interact with tools**
   - Use the Llama Stack playground UI for chatting with the agents
   - Use other frameworks and ecosystems like Claude Desktop, BeeAI, etc

---

## Resources for Continued Learning

### Official Documentation
- **[Docling Documentation](https://github.com/docling-project/docling)**: Latest features and updates

### Community Resources
- Join the Docling community on GitHub
- Share your implementations
- Contribute improvements back to the project

### Related Topics to Explore
- Document Layout Analysis
- Multimodal Embeddings
- Visual Question Answering
- Explainable AI Systems

---

## Final Thoughts

You've completed an incredible journey from basic document conversion to building a sophisticated, transparent AI system. The combination of Docling's document understanding with AI frameworks like Langchain, Llama Stack and MCP allows to build powerful applications.
